authors: id,name

categories: id, code, name

ingestion_runs: id, run_date, category_code, papers_fetched, papers_new, status, error_message,started_at, finished_at

paper_authors: author_id (key:authors[id]) paper_id (key:papers[id])

paper_categories: paper_id (key:papers[id]), category_id (key:categories[id]), is_primary

paper_embedings: paper_id (key:papers[id]), embedding (vector), model_used, created_at

paper_enrichments: id, paper_id (key:papers[id]), summary, methods, datasets, custom_topics, model_used, prompt_version, enriched_at

papers: id, arxiv_id, title, abstract, published_at, updated_at, pdf_url, ingested_at




In [ ]:
import urllib.parse
import requests
import time  # Added for rate limiting
from lxml import etree
base_url = "http://export.arxiv.org/api/query?{parameters}"
query_groups = {
    "nlp_and_core_ai": {"categories": ["cs.CL", "cs.AI"]},
    "ml_and_foundations": {"categories": ["cs.LG", "stat.ML"]},
    "multimodal_and_vision": {"categories": ["cs.CV"]},
    "safety_and_agents": {"categories": ["cs.CR", "cs.RO"]}
}

all_papers = []

for group_name, group_data in query_groups.items():
    print(f"Fetching daily update for: {group_name}")
    cat_string = " OR ".join([f"cat:{cat}" for cat in group_data["categories"]])
    
    parameters = urllib.parse.urlencode({
        "search_query": f"({cat_string}) AND all:LLM",
        "max_results": 100,  # Top 100 newest is plenty for a single day's volume
        "sortBy": "submittedDate",
        "sortOrder": "descending"
    })
    
    response = requests.get(base_url.format(parameters=parameters))
    root = etree.fromstring(response.content)
    namespaces = {'atom': 'http://www.w3.org/2005/Atom', 'arxiv': 'http://arxiv.org/schemas/atom'}

    entries = root.findall('atom:entry', namespaces)
    for entry in entries:
        arxiv_id = entry.find('atom:id', namespaces).text.split('/')[-1]
        title = entry.find('atom:title', namespaces).text.strip()
        abstract = entry.find('atom:summary', namespaces).text.strip()
        published_at = entry.find('atom:published', namespaces).text
        updated_at = entry.find('atom:updated', namespaces).text

        # Get the PDF URL
        pdf_url = None
        for link in entry.findall('atom:link', namespaces):
            if link.get('title') == 'pdf':
                pdf_url = link.get('href')
        # authors list into a comma-separated string for easier database insertion
        authors = ", ".join([author.text for author in entry.findall('atom:author/atom:name', namespaces)])
        
        cat_node = entry.find('arxiv:primary_category', namespaces)
        primary_cat = cat_node.get('term') if cat_node is not None else None
        
        all_categories = ", ".join([node.get('term') for node in entry.findall('atom:category', namespaces)])
        
        all_papers.append({
            "arxiv_id": arxiv_id,
            "title": title,
            "authors": authors,
            "abstract": abstract,
            "primary_cat": primary_cat,
            "all_categories": all_categories,
            "group_name": group_name,
            "link": pdf_url,
            "published_at": published_at,
            "updated_at": updated_at
        })
    
    time.sleep(3) # arXiv rate limits

print(f"Daily pull complete. Extracted {len(all_papers)} total papers.")


Fetching daily update for: nlp_and_core_ai
Fetching daily update for: ml_and_foundations
Fetching daily update for: multimodal_and_vision
Fetching daily update for: safety_and_agents
Daily pull complete. Extracted 400 total papers.


In [ ]:
import pandas as pd
        
df = pd.DataFrame(all_papers)

# remove any entire row where the arxiv_id, title, or abstract is missing
df = df.dropna(subset=['arxiv_id', 'title', 'abstract'])

# keeps the first from multiple rows with the exact same ID
df = df.drop_duplicates(subset=['arxiv_id'])

# normalize strings (ArXiv abstracts often contain hard line-breaks \n formatting the text for web display)
df['abstract'] = df['abstract'].str.replace(r'\s+', ' ', regex=True).str.strip()
df['title'] = df['title'].str.replace(r'\s+', ' ', regex=True).str.strip()

# Convert back to list of tuples for the Postgres insert hook
clean_papers = [tuple(x) for x in df.to_numpy()]


In [72]:
clean_papers

[('2607.02513v1',
  'LACUNA: A Testbed for Evaluating Localization Precision for LLM Unlearning',
  'Matteo Boglioni, Thibault Rousset, Siva Reddy, Marius Mosbach, Verna Dankers',
  "LLMs memorize sensitive training data, including personally identifiable information (PII), creating a pressing need for reliable post hoc removal methods. Unlearning has emerged as a promising solution, with state-of-the-art(SOTA) methods often following a localize-first, unlearn-second paradigm that targets specific model parameters. However, existing benchmarks evaluate unlearning solely at the output level, leaving open the question of whether unlearning truly erases knowledge from a model's parameters or merely obfuscates it, a concern reinforced by the success of resurfacing attacks. To bridge this gap, we introduce LACUNA: the first unlearning testbed with ground-truth parameter-level localization. LACUNA injects PII of synthetic individuals into predefined parameters of 1B and 7B OLMo-based models 

In [70]:
df_dropped

,arxiv_id,title,authors,abstract,primary_cat,all_categories,group_name
0,2607.02513v1,LACUNA: A Testbed for Evaluating Localization ...,"Matteo Boglioni, Thibault Rousset, Siva Reddy,...","LLMs memorize sensitive training data, includi...",cs.CL,"cs.CL, cs.AI, cs.LG",nlp_and_core_ai
1,2607.02510v1,Online Safety Monitoring for LLMs,"Mona Schirmer, Metod Jazbec, Alexander Timans,...","Despite alignment training, LLMs remain prone ...",cs.AI,"cs.AI, cs.CL, cs.LG, stat.AP, stat.ML",nlp_and_core_ai
2,2607.02509v1,ReContext: Recursive Evidence Replay as LLM Ha...,"Yanjun Zhao, Ruizhong Qiu, Tianxin Wei, Yuanch...",Understanding and reasoning over long contexts...,cs.AI,cs.AI,nlp_and_core_ai
3,2607.02507v1,What LLM Agents Say When No One Is Watching: S...,"Arman Ghaffarizadeh, Danyal Mohaddes, Aliakbar...",LLM agents will increasingly act in socially s...,cs.AI,"cs.AI, cs.CL, cs.LG, cs.MA",nlp_and_core_ai
4,2607.02504v1,Reasoning LLM Improves Speaker Recognition in ...,"Yuxuan Li, Lingxi Xie, Xinyue Huo, Jihao Qiu, ...",Long-form TV dramas present a formidable chall...,cs.CL,"cs.CL, cs.AI, cs.CV",nlp_and_core_ai
...,...,...,...,...,...,...,...
395,2606.25509v1,ASSCG: Just-Right Gating over Chattering for F...,"Sining Ang, Yuan Chen, Liu Haiyan, Xuanyao Mao...",Large language models (LLMs) can improve auton...,cs.RO,"cs.RO, cs.CV",safety_and_agents
396,2606.25497v1,SAGE-Nav: Leveraging LLM Planning and Alignmen...,"Hao Su, Yuehao Huang, Yukai Ma, Yong Liu, Jiaj...",Object-Goal Navigation (ObjNav) requires embod...,cs.RO,cs.RO,safety_and_agents
397,2606.25487v2,How Reliable Is Your Jailbreak Judge? Calibrat...,Yang Gao,Almost every paper on LLM jailbreaks and promp...,cs.CL,"cs.CL, cs.CR, cs.LG",safety_and_agents
398,2606.25404v1,HEART: Coordination of Heterogeneous Expert Ag...,"Junho Lee, Seabin Lee, Wonjong Lee, Nayoung Ki...",Large Language Models (LLMs) can reason over c...,cs.RO,cs.RO,safety_and_agents
